# Food Shelf-Life Prediction (Machine Learning Classification)

This notebook presents a machine learning approach to predict a food product shelf-life and product acceptability using physicochemical, microbiological, and sensory data.

---

**Note:**  
The dataset is not included due to confidentiality.  
This notebook demonstrates the workflow, methodology, and modeling approach used in the analysis.

# ========================================
0. Objective
1. Import libraries
2. Load dataset
3. Clean dataset
4. Dataset inspection
5. Define features and target
6. Feature engineering
7. Exploratory data analysis(EDA)
8. PCA
9. Exploratory Clustering
10. Model benchmark
11. Final train/test evaluation
12. Model interpretation
13. Conclusion
# ========================================

## 0. Objective

The objective of this project is to develop a machine learning model to predict food sample acceptability based on multiple quality indicators.

Specifically, this project aims to:

- Classify samples as acceptable or spoiled  
- Identify key factors driving spoilage  
- Explore relationships between microbial, chemical, and physical properties  
- Support a data-driven approach to shelf-life evaluation  

## 1. Import libraries

In [ ]:
# ==============================
# 1. Import libraries
# ==============================

# Data handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy.stats import pearsonr

# Preprocessing and dimensionality reduction
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.base import clone

# Clustering
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Model selection
from sklearn.model_selection import train_test_split, StratifiedKFold, StratifiedGroupKFold, cross_validate

# Classification models
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

# Pipeline
from sklearn.pipeline import Pipeline

# Hierarchical clustering visualization
from scipy.cluster.hierarchy import dendrogram, linkage

# SHAP (optional model interpretation)
import shap

# Display settings
pd.set_option("display.max_columns", None)

## 2. Load datase
### The dataset used in this project is not publicly available due to confidentiality.

### In practice, the data was loaded from an internal Excel file containing physicochemical, microbiological, and sensory measurements collected over storage time.

In [ ]:
# ==============================
# 2. Load dataset
# ==============================

# NOTE:
# The dataset is not included due to confidentiality.
# In the original analysis, data was loaded from an internal Excel file.

# Example (not executed here):
# df = pd.read_excel("path_to_dataset.xlsx")

# Placeholder to illustrate structure
df = pd.DataFrame()

### Dataset Description

The dataset consists of food samples stored under controlled conditions and analyzed over time.

- Storage period: 0 to 23 days  
- Storage temperatures: 0°C, 5°C, and 10°C  
- Sampling days: 0, 3, 7, 9, 13, 15, 17, 20, and 23  

### Feature categories:

**Physicochemical**
- pH  
- Color (L*, a*, b*)  

**Texture**
- Firmness, chewiness, adhesiveness, work of shear  

**Chemical indicators**
- TBARS, peroxide value, carbonyls, TVBN  

**Microbiological**
- Total plate count (TPC), LAB  

**Sensory**
- Odor score, color score  

**Volatile Compound Profile**
#
### Target column (Acceptability):
### 0 = Acceptable
### 1 = Rejected

## 3. Clean dataset

The dataset was prepared before modeling:

- Checked for missing values  
- Reviewed data types and consistency  
- Removed or excluded derived quality-label columns to avoid target leakage  
- Selected relevant input features  
- Defined the target variable for classification  

This step ensures that the model is trained on reliable and unbiased data.

In [ ]:
# ==============================
# 3. Clean column names
# ==============================

df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("*", "", regex=False)
)

df = df.rename(columns={
    "ButandIine_2_3": "Butanedione_2_3",
    "Butanoz_2": "Butanol_2",
    "Odour_sensory": "Odor_sensory",
    "Colour_sensory": "Color_sensory"
})

# Create a grouping identifier for triplicate measurements.
# This is used only for validation splitting, not as a model input.
# Triplicates from the same Day–Temperature condition should stay together.
if {"Day", "Temperature"}.issubset(df.columns):
    df["Group_ID"] = df["Day"].astype(str) + "_" + df["Temperature"].astype(str)


## 4. Dataset inspection

In [ ]:
# ==============================
# 4. Dataset inspection
# ==============================

display(df.head())
print("Shape:", df.shape)
df.info()
display(df.describe())
display(df.isnull().sum())

## 5. Define features and target

In [ ]:
# ==============================
# 5. Define metadata, features, and target
# ==============================

id_cols = [col for col in ["Sample_code", "Group_ID"] if col in df.columns]
storage_cols = ["Day", "Temperature"]
target_col = "Acceptability"

# Main feature set for EDA/correlation/modeling:
# includes Day and Temperature, excludes identifiers and target.
feature_cols_all = [col for col in df.columns if col not in id_cols + [target_col]]

# PCA/clustering feature set:
# same feature space as the main analysis at this stage.
feature_cols_pca_cluster = feature_cols_all.copy()

X_all = df[feature_cols_all].copy()
X_pca_cluster = df[feature_cols_pca_cluster].copy()
y = df[target_col].copy()

print("ID columns excluded:", id_cols)
print("Number of features for EDA/modeling:", len(feature_cols_all))
print("Number of features for PCA/clustering:", len(feature_cols_pca_cluster))
print("Acceptability in features?", target_col in X_all.columns)
print("Group_ID in features?", "Group_ID" in X_all.columns)
print("\nTarget distribution:")
display(y.value_counts())


## 6. Feature engineering

Feature engineering was applied to improve model performance and interpretation.

This included:

- Exploring relationships between variables  
- Identifying highly correlated features  
- Considering domain knowledge (e.g., microbial growth and oxidation processes)  
- Preparing features for machine learning models  

The goal was to capture meaningful patterns related to spoilage.

In [ ]:
# ==============================
# 6. Feature engineering
# ==============================

df["TBARS_to_PV"] = df["TBARS"] / (df["PV"] + 1e-6)

# Refresh feature sets after adding the engineered feature.
feature_cols_all = [col for col in df.columns if col not in id_cols + [target_col]]
feature_cols_pca_cluster = feature_cols_all.copy()

X_all = df[feature_cols_all].copy()
X_pca_cluster = df[feature_cols_pca_cluster].copy()
y = df[target_col].copy()

print("Updated number of features:", len(feature_cols_all))
print("Day included?", "Day" in X_all.columns)
print("Temperature included?", "Temperature" in X_all.columns)
print("TBARS_to_PV included?", "TBARS_to_PV" in X_all.columns)
print("Acceptability in X?", "Acceptability" in X_all.columns)
print("Group_ID in X?", "Group_ID" in X_all.columns)


## 7. Exploratory data analysis (EDA)

### 7.1 Correlation with target

In [ ]:
# ==============================
# 7.1 Correlation with target
# ==============================

# Use numeric predictors only for correlation analysis.
numeric_feature_cols = df[feature_cols_all].select_dtypes(include=np.number).columns.tolist()

target_corr = (
    df[numeric_feature_cols + [target_col]]
    .corr()[target_col]
    .sort_values(ascending=False)
)
display(target_corr)

target_corr_abs = target_corr.drop(target_col).abs().sort_values(ascending=False)
display(target_corr_abs.head(20))


### 7.2 Strongest feature-feature correlations

In [ ]:
# ==============================
# 7.2 Strongest feature-feature correlations
# ==============================

corr = df[numeric_feature_cols].corr()

corr_pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
corr_pairs = corr_pairs.stack().reset_index()
corr_pairs.columns = ["Feature_1", "Feature_2", "Correlation"]
corr_pairs["Abs_Correlation"] = corr_pairs["Correlation"].abs()

top_corr = corr_pairs.sort_values("Abs_Correlation", ascending=False)
display(top_corr.head(20))


### 7.3 Correlation heatmap
Correlation heatmap to explore relationships between variables
Helps identify highly correlated features and potential redundancy

In [ ]:
# ==============================
# 7.3 Correlation heatmap
# ==============================

top_features_for_corr = target_corr_abs.head(20).index.tolist()
corr_small = df[top_features_for_corr].corr()

def p_to_star(p):
    if p <= 0.0001:
        return "****"
    elif p <= 0.001:
        return "***"
    elif p <= 0.01:
        return "**"
    elif p <= 0.05:
        return "*"
    else:
        return ""

annot = corr_small.copy().astype(str)

for i, row_var in enumerate(top_features_for_corr):
    for j, col_var in enumerate(top_features_for_corr):
        r, p = pearsonr(df[row_var].astype(float), df[col_var].astype(float))
        annot.iloc[i, j] = f"{r:.2f}\n{p_to_star(p)}"

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_small, dtype=bool))
sns.heatmap(
    corr_small,
    mask=mask,
    annot=annot,
    fmt="",
    cmap="coolwarm",
    center=0,
    square=True
)
plt.title("Annotated Correlation Matrix (Top Features)")
plt.show()


### 7.4 Class distribution

In [ ]:
# ==============================
# 7.4 Class distribution
# ==============================

plt.figure(figsize=(5, 4))
sns.countplot(x=target_col, data=df)
plt.title("Acceptability Class Distribution")
plt.xlabel("Acceptability")
plt.ylabel("Count")
plt.show()

### 7.5 Selected scatter plots

In [ ]:
# ==============================
# 7.5 Selected scatter plots
# ==============================

scatter_pairs = [
    ("TPC_log", "LAB_log"),
    ("TPC_log", "TVBN"),
    ("TVBN", "Butanone_2"),
    ("PV", "TBARS"),
    ("Odor_sensory", "TVBN"),
    ("pentanal", "Hexanal"),
    ("TBARS_to_PV", "Acceptability")
]

for x_var, y_var in scatter_pairs:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(data=df, x=x_var, y=y_var, hue="Acceptability", s=80)
    plt.title(f"{y_var} vs {x_var}")
    plt.show()

### 7.6 Boxplots by target class

In [ ]:
# ==============================
# 7.6 Boxplots by target class
# ==============================

boxplot_features = [
    "TPC_log",
    "LAB_log",
    "TVBN",
    "Odor_sensory",
    "Color_L",
    "Firmness",
    "Stickiness",
    "Hexanol",
    "TBARS_to_PV"
]

for col in boxplot_features:
    plt.figure(figsize=(5, 4))
    sns.boxplot(data=df, x="Acceptability", y=col)
    plt.title(f"{col} by Acceptability")
    plt.show()

### 7.7 Trends over storage day

In [ ]:
# ==============================
# 7.7 Trends over storage day
# ==============================

trend_features = [
    "TPC_log",
    "TVBN",
    "Odor_sensory",
    "Color_L",
    "Firmness",
    "Hexanal",
    "TBARS_to_PV"
]

for col in trend_features:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(data=df, x="Day", y=col, hue="Temperature", style="Acceptability", s=90)
    plt.title(f"{col} vs Storage Day")
    plt.show()

### 7.8 Grouped distributions

In [ ]:
# ==============================
# 7.8 Grouped distributions (with KDE line)
# ==============================

grouped_features = {
    "Physicochemical": ["Color_L", "Color_a", "Color_b", "Firmness", "Work_of_shear", "Stickiness"],
    "Chemical": ["pH", "Carbonyl", "PV", "TBARS", "TVBN", "TBARS_to_PV"],
    "Microbiological": ["TPC_log", "LAB_log"],
    "Sensory": ["Odor_sensory", "Color_sensory"],
    "Volatile": ["Propanol_1", "Butandione_2_3", "Butanone_2", "Butanol_2", "Acetic_acid", "pentanal", "Propanoic_acid_2_oxo", "Hexanal", "Hexanol", "Ethanol_2_butoxy", "Benzaldehyde", "Benzeneacetaldehyde"]
}

for group_name, cols in grouped_features.items():
    
    n_cols = 3
    n_rows = (len(cols) + 2) // 3  # dynamic layout
    
    plt.figure(figsize=(14, 4 * n_rows))
    
    for i, col in enumerate(cols, 1):
        plt.subplot(n_rows, n_cols, i)
        sns.histplot(df[col], bins=15, kde=True)
        plt.title(col)
    
    plt.suptitle(f"{group_name} Feature Distributions", y=1.02)
    plt.tight_layout()
    plt.show()

## 8. PCA

### 8.1 Prepare datasets for PCA/clustering

#### PCA Strategy: With and Without Microbial Variables

PCA was performed using two feature sets:

- **With microbial variables** (e.g., TPC, LAB)
- **Without microbial variables**

Because microbial indicators were used to define acceptability, this comparison helps evaluate whether non-microbial features also capture spoilage patterns.

**Note:** PCA and clustering are exploratory visualization steps. Scaling for PCA/clustering is performed on the full dataset for visualization purposes only. Model evaluation later uses pipelines so scaling is fitted only on training folds/splits.


In [ ]:
# ==============================
# 8. Prepare datasets for PCA/clustering
# ==============================

X_cluster_all = X_pca_cluster.copy()
X_cluster_no_micro = X_pca_cluster.drop(
    columns=[c for c in ["TPC_log", "LAB_log"] if c in X_pca_cluster.columns]
).copy()

print("With microbial shape:", X_cluster_all.shape)
print("Without microbial shape:", X_cluster_no_micro.shape)
print("Acceptability in PCA X?", "Acceptability" in X_cluster_all.columns)
print("Group_ID in PCA X?", "Group_ID" in X_cluster_all.columns)


### 8.2 Scale datasets

In [ ]:
# ==============================
# 9. Scale datasets
# ==============================

scaler_all = StandardScaler()
X_cluster_all_scaled = pd.DataFrame(
    scaler_all.fit_transform(X_cluster_all),
    columns=X_cluster_all.columns,
    index=X_cluster_all.index
)

scaler_no_micro = StandardScaler()
X_cluster_no_micro_scaled = pd.DataFrame(
    scaler_no_micro.fit_transform(X_cluster_no_micro),
    columns=X_cluster_no_micro.columns,
    index=X_cluster_no_micro.index
)

### 8.3 PCA Analysis

PCA was applied to reduce dimensionality and visualize the structure of the data.

This helps to:

- Identify natural grouping of samples  
- Observe separation between fresh and spoiled samples  
- Understand how variables contribute to overall variation  

PCA provides insight into spoilage progression.

### 8.3.1 PCA — with microbial

In [ ]:
# ==============================
# 8.3.1 PCA — WITH microbial
# ==============================

pca_all = PCA(n_components=3, random_state=42)
X_pca_all = pca_all.fit_transform(X_cluster_all_scaled)

df_pca_all = pd.DataFrame(X_pca_all, columns=["PC1", "PC2", "PC3"], index=df.index)
df_pca_all["Acceptability"] = df["Acceptability"]
df_pca_all["Day"] = df["Day"]
df_pca_all["Temperature"] = df["Temperature"]

print("Explained variance ratio:", pca_all.explained_variance_ratio_)
print("Total variance explained (3 PCs):", pca_all.explained_variance_ratio_.sum())


### 8.3.2 PCA — without microbial

In [ ]:
# ==============================
# 8.3.2 PCA — WITHOUT microbial
# ==============================

pca_no_micro = PCA(n_components=3, random_state=42)
X_pca_no_micro = pca_no_micro.fit_transform(X_cluster_no_micro_scaled)

df_pca_no_micro = pd.DataFrame(X_pca_no_micro, columns=["PC1", "PC2", "PC3"], index=df.index)
df_pca_no_micro["Acceptability"] = df["Acceptability"]
df_pca_no_micro["Day"] = df["Day"]
df_pca_no_micro["Temperature"] = df["Temperature"]

print("Explained variance ratio:", pca_no_micro.explained_variance_ratio_)
print("Total variance explained (3 PCs):", pca_no_micro.explained_variance_ratio_.sum())


### 8.4 2D PCA plots with percentages

In [ ]:
# ==============================
# 8.4 PCA 2D plots
# ==============================

plt.figure(figsize=(7,5))
sns.scatterplot(data=df_pca_all, x="PC1", y="PC2", hue="Acceptability", palette="coolwarm", s=100)
plt.xlabel(f"PC1 ({pca_all.explained_variance_ratio_[0]*100:.2f}%)")
plt.ylabel(f"PC2 ({pca_all.explained_variance_ratio_[1]*100:.2f}%)")
plt.title("PCA Projection (WITH Microbial)")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

plt.figure(figsize=(7,5))
sns.scatterplot(data=df_pca_no_micro, x="PC1", y="PC2", hue="Acceptability", palette="coolwarm", s=100)
plt.xlabel(f"PC1 ({pca_no_micro.explained_variance_ratio_[0]*100:.2f}%)")
plt.ylabel(f"PC2 ({pca_no_micro.explained_variance_ratio_[1]*100:.2f}%)")
plt.title("PCA Projection (WITHOUT Microbial)")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 8.5 PCA comparison

In [ ]:
# ==============================
# 8.5 PCA comparison
# ==============================

comparison_pca = pd.DataFrame({
    "PC1_variance": [
        pca_all.explained_variance_ratio_[0],
        pca_no_micro.explained_variance_ratio_[0]
    ],
    "PC2_variance": [
        pca_all.explained_variance_ratio_[1],
        pca_no_micro.explained_variance_ratio_[1]
    ],
    "Total_variance": [
        pca_all.explained_variance_ratio_.sum(),
        pca_no_micro.explained_variance_ratio_.sum()
    ]
}, index=["With_microbial", "Without_microbial"])

display(comparison_pca)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))

sns.scatterplot(
    data=df_pca_all, x="PC1", y="PC2",
    hue="Acceptability", palette="coolwarm", s=100, ax=axes[0]
)
axes[0].set_title("WITH Microbial Features")

sns.scatterplot(
    data=df_pca_no_micro, x="PC1", y="PC2",
    hue="Acceptability", palette="coolwarm", s=100, ax=axes[1]
)
axes[1].set_title("WITHOUT Microbial Features")

for ax in axes:
    ax.axhline(0, linestyle="--", color="gray")
    ax.axvline(0, linestyle="--", color="gray")

plt.tight_layout()
plt.show()

### 8.6 Principal component  1 & 2

In [ ]:
#### 8.6 Principal component  1 & 2
#PCA loadings (WITH microbial)
loadings_all = pd.DataFrame(
    pca_all.components_.T,
    columns=["PC1", "PC2", "PC3"],
    index=X_cluster_all.columns
)

# Top contributing features
top_pc1 = loadings_all["PC1"].abs().sort_values(ascending=False)
top_pc2 = loadings_all["PC2"].abs().sort_values(ascending=False)

print("Top features contributing to PC1:")
display(top_pc1.head(10))

print("Top features contributing to PC2:")
display(top_pc2.head(10))

# Direction of contribution (interpretation)
pc1_sorted = loadings_all["PC1"].sort_values(ascending=False)

print("Top positive contributors (PC1):")
display(pc1_sorted.head(5))

print("Top negative contributors (PC1):")
display(pc1_sorted.tail(5))

## 9. Exploratory Clustering Analysis

Clustering was used as an exploratory step to investigate whether samples naturally group according to spoilage status, storage time, or temperature. It was not used as the final predictive model.

### 9.1 KMeans k=2 — with microbial

In [ ]:
# ==============================
# 9.1 KMeans k=2 — WITH microbial
# ==============================

kmeans_all = KMeans(n_clusters=2, random_state=42)
clusters_all = kmeans_all.fit_predict(X_cluster_all_scaled)

df_pca_all["Cluster_k2"] = clusters_all

display(pd.crosstab(df_pca_all["Cluster_k2"], df_pca_all["Acceptability"]))

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df_pca_all, x="PC1", y="PC2", hue="Cluster_k2", palette="Set1", s=100)
plt.title("KMeans (k=2) WITH Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 9.2 KMeans k=2 — without microbial

In [ ]:
# ==============================
# 9.2 KMeans k=2 — WITHOUT microbial
# ==============================

kmeans_no_micro = KMeans(n_clusters=2, random_state=42)
clusters_no_micro = kmeans_no_micro.fit_predict(X_cluster_no_micro_scaled)

df_pca_no_micro["Cluster_k2"] = clusters_no_micro

display(pd.crosstab(df_pca_no_micro["Cluster_k2"], df_pca_no_micro["Acceptability"]))

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df_pca_no_micro, x="PC1", y="PC2", hue="Cluster_k2", palette="Set1", s=100)
plt.title("KMeans (k=2) WITHOUT Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 9.3 KMeans k=3 — with microbial

In [ ]:
# ==============================
# 9.3 KMeans k=3 — WITH microbial
# ==============================

kmeans_all_3 = KMeans(n_clusters=3, random_state=42)
clusters_all_3 = kmeans_all_3.fit_predict(X_cluster_all_scaled)

df_pca_all["Cluster_k3"] = clusters_all_3
df["Cluster_3_with_micro"] = clusters_all_3

cluster_table_all_3 = pd.crosstab(
    pd.Series(clusters_all_3, name="Cluster"),
    df["Acceptability"]
)

display(cluster_table_all_3)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df_pca_all, x="PC1", y="PC2", hue="Cluster_k3", palette="Set2", s=100)
plt.title("KMeans (k=3) WITH Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 9.4 KMeans k=3 — without microbial

In [ ]:
# ==============================
# 9.4 KMeans k=3 — WITHOUT microbial
# ==============================

kmeans_no_micro_3 = KMeans(n_clusters=3, random_state=42)
clusters_no_micro_3 = kmeans_no_micro_3.fit_predict(X_cluster_no_micro_scaled)

df_pca_no_micro["Cluster_k3"] = clusters_no_micro_3
df["Cluster_3_no_micro"] = clusters_no_micro_3

cluster_table_no_micro_3 = pd.crosstab(
    pd.Series(clusters_no_micro_3, name="Cluster"),
    df["Acceptability"]
)

display(cluster_table_no_micro_3)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df_pca_no_micro, x="PC1", y="PC2", hue="Cluster_k3", palette="Set2", s=100)
plt.title("KMeans (k=3) WITHOUT Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 9.5 Hierarchical clustering — with microbial

In [ ]:
# ==============================
# 9.5 Hierarchical clustering — WITH microbial
# ==============================

agg_all = AgglomerativeClustering(n_clusters=2)
clusters_hier_all = agg_all.fit_predict(X_cluster_all_scaled)

df_pca_all["Cluster_hier"] = clusters_hier_all

cluster_table_agg_all = pd.crosstab(
    pd.Series(clusters_hier_all, name="Cluster"),
    df["Acceptability"]
)

display(cluster_table_agg_all)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df_pca_all, x="PC1", y="PC2", hue="Cluster_hier", palette="Set1", s=100)
plt.title("Hierarchical Clustering WITH Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 9.6 Hierarchical clustering — without microbial

In [ ]:
# ==============================
# 9.6 Hierarchical clustering — WITHOUT microbial
# ==============================

agg_no_micro = AgglomerativeClustering(n_clusters=2)
clusters_hier_no_micro = agg_no_micro.fit_predict(X_cluster_no_micro_scaled)

df_pca_no_micro["Cluster_hier"] = clusters_hier_no_micro

cluster_table_agg_no_micro = pd.crosstab(
    pd.Series(clusters_hier_no_micro, name="Cluster"),
    df["Acceptability"]
)

display(cluster_table_agg_no_micro)

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(data=df_pca_no_micro, x="PC1", y="PC2", hue="Cluster_hier", palette="Set1", s=100)
plt.title("Hierarchical Clustering WITHOUT Microbial")
plt.axhline(0, linestyle="--", color="gray")
plt.axvline(0, linestyle="--", color="gray")
plt.show()

### 9.7 Silhouette comparison

In [ ]:
# ==============================
# 9.7 Silhouette comparison
# ==============================

sil_k2_all = silhouette_score(X_cluster_all_scaled, clusters_all)
sil_k2_no = silhouette_score(X_cluster_no_micro_scaled, clusters_no_micro)

sil_k3_all = silhouette_score(X_cluster_all_scaled, clusters_all_3)
sil_k3_no = silhouette_score(X_cluster_no_micro_scaled, clusters_no_micro_3)

sil_hier_all = silhouette_score(X_cluster_all_scaled, clusters_hier_all)
sil_hier_no = silhouette_score(X_cluster_no_micro_scaled, clusters_hier_no_micro)

sil_summary = pd.DataFrame({
    "Model": [
        "KMeans k=2 (with micro)",
        "KMeans k=2 (no micro)",
        "KMeans k=3 (with micro)",
        "KMeans k=3 (no micro)",
        "Hierarchical (with micro)",
        "Hierarchical (no micro)"
    ],
    "Silhouette Score": [
        sil_k2_all,
        sil_k2_no,
        sil_k3_all,
        sil_k3_no,
        sil_hier_all,
        sil_hier_no
    ]
})

display(sil_summary)

## 10. Classification modeling benchmark

### 10.1 Define target and feature sets

In [ ]:
# ==============================
# 10.1 Define target and feature sets
# ==============================

cluster_cols = [col for col in df.columns if "Cluster" in col]
id_cols_model = [col for col in ["Sample_code", "Group_ID"] if col in df.columns]

y_model = df["Acceptability"].copy()

# WITH microbial + Day + Temperature + engineered features
X_model_all = df.drop(columns=id_cols_model + ["Acceptability"] + cluster_cols)

# WITHOUT microbial + Day + Temperature + engineered features
X_model_no_micro = X_model_all.drop(
    columns=[c for c in ["TPC_log", "LAB_log"] if c in X_model_all.columns]
)

print("Target distribution:")
print(y_model.value_counts())

print("\nShape with microbial + storage:", X_model_all.shape)
print("Shape without microbial + storage:", X_model_no_micro.shape)
print("Acceptability in X?", "Acceptability" in X_model_all.columns)
print("Sample_code in X?", "Sample_code" in X_model_all.columns)
print("Group_ID in X?", "Group_ID" in X_model_all.columns)
print("Cluster columns removed:", cluster_cols)


### 10.2 Cross-validation setup

In [ ]:
# ==============================
# 10.2 Cross-validation setup
# ==============================

# Random validation baseline with group-aware stratification.
# Group_ID keeps triplicate measurements from the same Day–Temperature condition together.
groups = df["Group_ID"]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}


### 10.3 Models

In [ ]:
# ==============================
# 10.3 Benchmark models
# ==============================


models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    "Ridge Classifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", RidgeClassifier())
    ]),
    "SGD Classifier": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SGDClassifier(random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True))
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier())
    ]),
    "Nearest Centroid": Pipeline([
        ("scaler", StandardScaler()),
        ("model", NearestCentroid())
    ]),
    "Linear Discriminant Analysis": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearDiscriminantAnalysis())
    ]),
    "Gaussian NB": GaussianNB(),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        eval_metric="logloss"
    )
}

### 10.4 Evaluation function

Models were evaluated using:

- Accuracy  
- Precision, recall, and F1-score  
- Confusion matrix  

Special attention was given to correctly identifying spoiled samples, as misclassification could have practical consequences in food safety.

In [ ]:
# ==============================
# 10.4 Evaluation function
# ==============================

def evaluate_models(X, y, models, cv, scoring, groups=None):
    results = []

    for name, model in models.items():
        cv_results = cross_validate(
            model,
            X,
            y,
            cv=cv,
            groups=groups,
            scoring=scoring,
            return_train_score=False
        )

        results.append({
            "Model": name,
            "Accuracy": cv_results["test_accuracy"].mean(),
            "Precision": cv_results["test_precision"].mean(),
            "Recall": cv_results["test_recall"].mean(),
            "F1 Score": cv_results["test_f1"].mean()
        })

    return pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)


### 10.5 Run models — with microbial

In [ ]:
# ==============================
# 10.5 Run models — WITH microbial
# ==============================

results_with_micro = evaluate_models(
    X_model_all,
    y_model,
    models,
    cv,
    scoring,
    groups=groups
)

display(results_with_micro)


### 10.6 Run models — without microbial

In [ ]:
# ==============================
# 10.6 Run models — WITHOUT microbial
# ==============================

results_no_micro = evaluate_models(
    X_model_no_micro,
    y_model,
    models,
    cv,
    scoring,
    groups=groups
)

display(results_no_micro)


### 10.7 Benchmark plots

In [ ]:
# ==============================
# 10.7 Benchmark plots
# ==============================

plt.figure(figsize=(10,6))
sns.barplot(data=results_with_micro, y="Model", x="F1 Score")
plt.title("Model Benchmark (WITH Microbial)")
plt.show()

plt.figure(figsize=(10,6))
sns.barplot(data=results_no_micro, y="Model", x="F1 Score")
plt.title("Model Benchmark (WITHOUT Microbial)")
plt.show()

## 11. Final train/test evaluation

### Important Note on Validation

This notebook uses random train/test splitting as a baseline evaluation.  
Because shelf-life data has a temporal structure, this approach may overestimate model performance.  
A separate time-aware validation notebook was developed to evaluate the model under more realistic conditions.

### 11.1 Select final 4 models

In [ ]:
# ==============================
# 11.1 Final selected models
# ==============================

selected_models = {
    "Logistic Regression": models["Logistic Regression"],
    "Random Forest": models["Random Forest"],
    "Gradient Boosting": models["Gradient Boosting"],
    "XGBoost": models["XGBoost"]
}

### 11.2 Train/test split

In [ ]:
# ==============================
# 11.2 Train/test split — STRATIFIED GROUP SPLIT
# ==============================

# Use one split from StratifiedGroupKFold to keep triplicates together
# while preserving class balance as much as possible.
groups = df["Group_ID"]

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(sgkf.split(X_model_all, y_model, groups))

X_train_with = X_model_all.iloc[train_idx].copy()
X_test_with = X_model_all.iloc[test_idx].copy()
y_train_with = y_model.iloc[train_idx].copy()
y_test_with = y_model.iloc[test_idx].copy()

X_train_no = X_model_no_micro.iloc[train_idx].copy()
X_test_no = X_model_no_micro.iloc[test_idx].copy()
y_train_no = y_model.iloc[train_idx].copy()
y_test_no = y_model.iloc[test_idx].copy()

print("Train distribution:")
print(y_train_with.value_counts())

print("\nTest distribution:")
print(y_test_with.value_counts())

print("\nTest set day x acceptability:")
display(pd.crosstab(df.iloc[test_idx]["Day"], df.iloc[test_idx]["Acceptability"]))

print("\nTest set group check:")
print(df.iloc[test_idx]["Group_ID"].value_counts())


### 11.3 Train selected models

In [ ]:
# ==============================
# 11.3 Train selected models
# ==============================

trained_models_with = {}
trained_models_no = {}

for name, model in selected_models.items():
    model_copy_with = clone(model)
    model_copy_with.fit(X_train_with, y_train_with)
    trained_models_with[name] = model_copy_with

    model_copy_no = clone(model)
    model_copy_no.fit(X_train_no, y_train_no)
    trained_models_no[name] = model_copy_no

### 11.4 Test results

In [ ]:
# ==================================
# 11.4 Test results
# ==================================

test_results_with = []
for name, model in trained_models_with.items():
    y_pred = model.predict(X_test_with)
    test_results_with.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_with, y_pred),
        "Precision": precision_score(y_test_with, y_pred),
        "Recall": recall_score(y_test_with, y_pred),
        "F1 Score": f1_score(y_test_with, y_pred)
    })

test_results_with_df = pd.DataFrame(test_results_with).sort_values(by="F1 Score", ascending=False)
display(test_results_with_df)

test_results_no = []
for name, model in trained_models_no.items():
    y_pred = model.predict(X_test_no)
    test_results_no.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_no, y_pred),
        "Precision": precision_score(y_test_no, y_pred),
        "Recall": recall_score(y_test_no, y_pred),
        "F1 Score": f1_score(y_test_no, y_pred)
    })

test_results_no_df = pd.DataFrame(test_results_no).sort_values(by="F1 Score", ascending=False)
display(test_results_no_df)

### 11.5 Classification reports

In [ ]:
# ============================================
# 11.5 Classification reports- WITH microbial
# ============================================

for name, model in trained_models_with.items():
    y_pred = model.predict(X_test_with)
    print(f"\n{name} — WITH microbial")
    print(classification_report(y_test_with, y_pred, zero_division=0))




In [ ]:
# ============================================
# 11.5 Classification reports- WITHOUT microbial
# ============================================

for name, model in trained_models_no.items():
    y_pred = model.predict(X_test_no)
    print(f"\n{name} — WITHOUT microbial")
    print(classification_report(y_test_no, y_pred, zero_division=0))

### 11.6 Confusion matrices

In [ ]:
# ========================================
# 11.6 Confusion matrices- WITH microbial
# ========================================

for name, model in trained_models_with.items():
    y_pred = model.predict(X_test_with)
    ConfusionMatrixDisplay.from_predictions(y_test_with, y_pred)
    plt.title(f"{name} — WITH microbial")
    plt.show()

In [ ]:
# ========================================
# 11.6 Confusion matrices- WITHOUT microbial
# ========================================

for name, model in trained_models_no.items():
    y_pred = model.predict(X_test_no)
    ConfusionMatrixDisplay.from_predictions(y_test_no, y_pred)
    plt.title(f"{name} — WITHOUT microbial")
    plt.show()

## 12. Model interpretation

### 12.1 Feature importance

#### 12.1.1 Random Forest feature importance

In [ ]:
# =========================================
# 12.1.1 Feature importance - Random Forest
# =========================================

rf_with = trained_models_with["Random Forest"]
rf_no = trained_models_no["Random Forest"]

rf_importance_with = pd.Series(rf_with.feature_importances_, index=X_train_with.columns).sort_values(ascending=False)
rf_importance_no = pd.Series(rf_no.feature_importances_, index=X_train_no.columns).sort_values(ascending=False)

display(rf_importance_with.head(15))
display(rf_importance_no.head(15))

In [ ]:
plt.figure(figsize=(8,6))
rf_importance_with.head(15).sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance - WITH microbial")
plt.xlabel("Importance")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
rf_importance_no.head(15).sort_values().plot(kind="barh")
plt.title("Random Forest Feature Importance - WITHOUT microbial")
plt.xlabel("Importance")
plt.show()

#### 12.1.2 Gradient Boosting feature importance

In [ ]:
# =============================================
# 12.1.2 Feature importance - Gradient Boosting
# =============================================

gb_with = trained_models_with["Gradient Boosting"]
gb_no = trained_models_no["Gradient Boosting"]

gb_importance_with = pd.Series(gb_with.feature_importances_, index=X_train_with.columns).sort_values(ascending=False)
gb_importance_no = pd.Series(gb_no.feature_importances_, index=X_train_no.columns).sort_values(ascending=False)

display(gb_importance_with.head(15))
display(gb_importance_no.head(15))

In [ ]:
plt.figure(figsize=(8,6))
gb_importance_with.head(32).sort_values().plot(kind="barh")
plt.title("Gradient Boosting Feature Importance - WITH microbial")
plt.xlabel("Importance")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
gb_importance_no.head(15).sort_values().plot(kind="barh")
plt.title("Gradient Boosting Feature Importance - WITHOUT microbial")
plt.xlabel("Importance")
plt.show()

### 12.2 Logistic Regression coefficients

In [ ]:
# ======================================
# 12.2 Logistic Regression coefficients
# ======================================

logreg_with = trained_models_with["Logistic Regression"]
logreg_no = trained_models_no["Logistic Regression"]

coef_with = pd.Series(
    logreg_with.named_steps["model"].coef_[0],
    index=X_train_with.columns
).sort_values(key=np.abs, ascending=False)

coef_no = pd.Series(
    logreg_no.named_steps["model"].coef_[0],
    index=X_train_no.columns
).sort_values(key=np.abs, ascending=False)

display(coef_with.head(15))
display(coef_no.head(15))

In [ ]:
plt.figure(figsize=(8,6))
coef_with.head(20).sort_values().plot(kind="barh")
plt.title("Logistic Regression Coefficients - WITH microbial")
plt.xlabel("Coefficient")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
coef_no.head(15).sort_values().plot(kind="barh")
plt.title("Logistic Regression Coefficients - WITHOUT microbial")
plt.xlabel("Coefficient")
plt.show()

### 12.3 Optional SHAP analysis

SHAP can provide detailed model interpretation, but it is optional and can be computationally heavier. For the main presentation, feature importance and logistic regression coefficients were prioritized for clarity.


#### 12.3.1 SHAP for Logistic Regression

In [ ]:
# ===================================================
# 12.3.1 SHAP for Logistic Regression- WITH microbial
# ===================================================

logreg_model_with = trained_models_with["Logistic Regression"]
scaler_with = logreg_model_with.named_steps["scaler"]
lr_with = logreg_model_with.named_steps["model"]

X_lr_with_scaled = pd.DataFrame(
    scaler_with.transform(X_train_with),
    columns=X_train_with.columns,
    index=X_train_with.index
)

explainer_lr_with = shap.LinearExplainer(lr_with, X_lr_with_scaled)
shap_values_lr_with = explainer_lr_with.shap_values(X_lr_with_scaled)

shap.summary_plot(shap_values_lr_with, X_lr_with_scaled, plot_type="bar")
shap.summary_plot(shap_values_lr_with, X_lr_with_scaled)

In [ ]:
# ======================================================
# 14.3.2 SHAP for Logistic Regression- WITHOUT microbial
# ======================================================

logreg_model_no = trained_models_no["Logistic Regression"]
scaler_no = logreg_model_no.named_steps["scaler"]
lr_no = logreg_model_no.named_steps["model"]

X_lr_no_scaled = pd.DataFrame(
    scaler_no.transform(X_train_no),
    columns=X_train_no.columns,
    index=X_train_no.index
)

explainer_lr_no = shap.LinearExplainer(lr_no, X_lr_no_scaled)
shap_values_lr_no = explainer_lr_no.shap_values(X_lr_no_scaled)

shap.summary_plot(shap_values_lr_no, X_lr_no_scaled, plot_type="bar")
shap.summary_plot(shap_values_lr_no, X_lr_no_scaled)

#### 12.3.3 SHAP for Random Forest

In [ ]:
# ==============================================
# 12.3.3 SHAP for random forest— WITH microbial
# ==============================================

rf_model_with = trained_models_with["Random Forest"]
X_rf_with = X_train_with.copy()

# Build SHAP explanation
explainer_rf_with = shap.Explainer(rf_model_with, X_rf_with)
rf_exp_with = explainer_rf_with(X_rf_with)

print("RF explanation shape:", rf_exp_with.values.shape)

In [ ]:
# Select class 1 = Rejected
rf_exp_with_reject = rf_exp_with[:, :, 1]

# Summary bar plot
shap.plots.bar(rf_exp_with_reject, max_display=15)

# Beeswarm summary plot
shap.plots.beeswarm(rf_exp_with_reject, max_display=31)

In [ ]:
rf_shap_with_importance = pd.Series(
    np.abs(rf_exp_with_reject.values).mean(axis=0),
    index=X_rf_with.columns
).sort_values(ascending=False)

display(rf_shap_with_importance.head(31))

In [ ]:
# ==========================================
# 12.3.4 SHAP for Random Forest— WITHOUT microbial
# ==========================================

rf_model_no = trained_models_no["Random Forest"]
X_rf_no = X_train_no.copy()

explainer_rf_no = shap.Explainer(rf_model_no, X_rf_no)
rf_exp_no = explainer_rf_no(X_rf_no)

print("RF explanation shape (no micro):", rf_exp_no.values.shape)

In [ ]:
rf_exp_no_reject = rf_exp_no[:, :, 1]

shap.plots.bar(rf_exp_no_reject, max_display=15)
shap.plots.beeswarm(rf_exp_no_reject, max_display=31)

In [ ]:
rf_vals_no = rf_exp_no.values

if rf_vals_no.ndim == 3:
    rf_vals_no = rf_vals_no[:, :, 1]

rf_shap_no_importance = pd.Series(
    np.abs(rf_vals_no).mean(axis=0),
    index=X_rf_no.columns
).sort_values(ascending=False)

display(rf_shap_no_importance.head(15))

### 12.4 Compare important variables across interpretation methods

In [ ]:
# ===========================================
# 12.4 Compare top variables — WITH microbial
# ===========================================

comparison_with_dict = {
    "LogReg_coef": coef_with.head(10).index,
    "RF_importance": rf_importance_with.head(10).index,
    "GB_importance": gb_importance_with.head(10).index
}

if "rf_shap_with_importance" in globals():
    comparison_with_dict["RF_SHAP"] = rf_shap_with_importance.head(10).index

comparison_with = pd.DataFrame(comparison_with_dict)
display(comparison_with)


In [ ]:
# ==============================================
# 12.4 Compare top variables — WITHOUT microbial
# ==============================================

comparison_no_dict = {
    "LogReg_coef": coef_no.head(10).index,
    "RF_importance": rf_importance_no.head(10).index,
    "GB_importance": gb_importance_no.head(10).index
}

if "rf_shap_no_importance" in globals():
    comparison_no_dict["RF_SHAP"] = rf_shap_no_importance.head(10).index

comparison_no = pd.DataFrame(comparison_no_dict)
display(comparison_no)


### 12.5 Final comparison tables

#### 12.5.1 Benchmark summary

In [ ]:
# ==============================
# 12.5.1 Final benchmark summary
# ==============================

final_benchmark_summary = pd.DataFrame({
    "WITH_micro_CV_F1": results_with_micro.set_index("Model")["F1 Score"],
    "WITHOUT_micro_CV_F1": results_no_micro.set_index("Model")["F1 Score"]
})

display(final_benchmark_summary.sort_values(by="WITH_micro_CV_F1", ascending=False))

#### 12.5.2 Test summary

In [ ]:
# ==============================
# 12.5.2 Final test summary
# ==============================

final_test_summary = pd.DataFrame({
    "WITH_micro_test_F1": test_results_with_df.set_index("Model")["F1 Score"],
    "WITHOUT_micro_test_F1": test_results_no_df.set_index("Model")["F1 Score"]
})

display(final_test_summary.sort_values(by="WITH_micro_test_F1", ascending=False))

### Final interpretation notes

1. Compare benchmark performance with and without microbial variables.
2. Interpret random validation as a baseline; time-aware validation provides a more realistic future-prediction scenario.
3. Use test results only as supporting evidence due to the small sample size.
4. Compare which variables repeatedly appear across:
    - Logistic Regression coefficients
    - Random Forest importance
    - Gradient Boosting importance
    - Optional SHAP values
5. Discuss whether microbial variables dominate prediction or whether sensory, physicochemical, and storage-condition variables can serve as practical alternatives.


## 13. Conclusion
### Overall, this baseline notebook shows that machine learning can classify food product acceptability using combined food quality indicators. However, because random splitting may not fully reflect real shelf-life prediction, results should be interpreted as baseline performance. The time-aware validation notebook provides a more realistic evaluation of model generalization to later storage days.